In [1]:
from lcpy.calculators.helpers import ExchangeHolder, ImpactCalculator, ImpactHandler
from lcpy.calculators.bw_int import mpLCAer
import os
from lcpy.hvs.hvs import create_dataframe_dict, save_dataframes_to_excel, create_name_dictionaries,plot_stacked_percentage_bar_grid, plot_stacked_percentage_bar_sub_processes, plot_stacked_percentage_barchart_seaborn
from lcpy.hvs.map_dicts import create_mapping, create_list_with_unique_activities
from lcpy.hvs.hvs import make_characterized_inventory_dfs_simple_lca
from lcpy.calculators.env_calc import fast_calculator


# Simple LCA with two main processes 
- recipe production
- transportation

## Define bw configuration

In [51]:
target_dir = "results/lca"
os.makedirs(target_dir, exist_ok=True)

# define environmental impact calculation methods (e.g. TRACI v2.1)
methods_list = [
('TRACI v2.1', 'acidification', 'acidification potential (AP)'),
('TRACI v2.1', 'climate change', 'global warming potential (GWP100)'),
('TRACI v2.1', 'ecotoxicity: freshwater', 'ecotoxicity: freshwater'),
('TRACI v2.1', 'eutrophication', 'eutrophication potential'),
('TRACI v2.1', 'human toxicity: carcinogenic', 'human toxicity: carcinogenic'),
('TRACI v2.1', 'human toxicity: non-carcinogenic', 'human toxicity: non-carcinogenic'),
('TRACI v2.1', 'ozone depletion', 'ozone depletion potential (ODP)'),
('TRACI v2.1', 'particulate matter formation', 'particulate matter formation potential (PMFP)'),
('TRACI v2.1', 'photochemical oxidant formation', 'maximum incremental reactivity (MIR)'),
]
method_units_list = ['kg SO2-Eq',
 'kg CO2-Eq',
 'CTUe',
 'kg N-Eq',
 'CTUh',
 'CTUh',
 'kg CFC-11-Eq',
 'kg PM2.5-Eq',
 'kg O3-Eq',
]

# link to brightway - folder name for bw environment, project name, database names, etc. 
brightway_configuration_dictionary = {
    # path_to_folder_containing_the_bw_environment_and_packages_installed_there
    "path_to_brightway_project": "/Users/tsuitpy/Library/Application Support/Brightway3/biopol_lca.967046090aaaeb21fcd35d8654c4df5a", 
    "bw_project": "biopol_lca",
    "bw_database": "biopol_lca_database", 
    "bw_biosphere": "ecoinvent-3.9.1-biosphere",
    "bw_ecoinvent": "ecoinvent-3.9.1-cutoff"
}
methods_gp = methods_list[:]
impact_categories_names = ['AP', 'GWP100', 'ECFW', 'EP', 'HTC', 'HTNC', 'ODP', 'PMFP', 'MIR']
impact_categories_units = method_units_list
scenarios = 100
timeframe = 1 #operational lifetime after construction
time_step = 1
construction_years = 0
number_of_infrastructure_processes = 0
# scenario_names = ['one']
# timeframe = 2 #operational lifetime after construction
# time_step = 1
# construction_years = 0


## Define keys

In [52]:
keys_binder = {
    'b1_potato_starch_mass' : '8ce68e5f71f8f8c5dc50edb913e9cea1', # market for potato starch ('ecoinvent-3.9.1-cutoff', '8ce68e5f71f8f8c5dc50edb913e9cea1')
    'b2_lignin_resin_mass': '9ef63345ae414b0aa5683f09f4d0b806', # 'market for anionic resin' (kilogram, RER, None)
    'b3_soy_resin_mass': '16a6869a94ad2aaf84d113c88657406b' # 'market for phenolic resin' (kilogram, RER, None) 
}
keys_filler = {
    'f1_wood_fiber_mass' : 'd82e94ec45e662362de31d0ac344dc51', # 'market for wood wool' (kilogram, RER, None) 
    'f2_hemp_shiv_mass': 'b8b0f0ca4e8e1a99ddae3f6ae610be2e', # 'market for sunn hemp plant, harvested' (kilogram, GLO, None)
    'f3_rice_husk_mass': '94c1cb9e760faf73987ffea6c08cfcf1' # 'market for coconut husk' (kilogram, GLO, None) 
}
keys_recipe = {
    'binder': '', 
    'filler': '' 
}

## Define exchange amounts

Here's where we define the amounts for the biopolymer recipe. This determines how much we should multiply the unit impacts (pre-calculated in the previous section by `mpLCAer`) by. 

In [81]:
# DEFINE ABSOLOUTE AMOUNTS FOR SUB-SUB-, SUB-, AND MAIN PROCESS

# sub-sub-processes: values to be adjusted according to recipe 
b1_potato_starch_mass = 40
b2_lignin_resin_mass = 50
b3_soy_resin_mass = 10
f1_wood_fiber_mass = 60
f2_hemp_shiv_mass = 30
f3_rice_husk_mass = 10 

# sub-processes 
binder_mass = b1_potato_starch_mass + b2_lignin_resin_mass + b3_soy_resin_mass
filler_mass = f1_wood_fiber_mass + f2_hemp_shiv_mass + f3_rice_husk_mass 

# main process 
total_mass = binder_mass + filler_mass

In [82]:
# DEFINE EXCHANGE AMOUNTS BETWEEN THE PROCESSES 

# sub-processes 
binder_exchanges_amounts = [ 
    # proportion of binder ingredients (b1, b2, b3) in the binder
    # looks something like this: [1.0, 0.0, 0.0]
    # order of the amounts matches with keys_binder 
    b1_potato_starch_mass / binder_mass, 
    b2_lignin_resin_mass / binder_mass, 
    b3_soy_resin_mass / binder_mass
] 
filler_exchanges_amounts = [
    # proportion of filler ingredients (b1, b2, b3) in the filler
    # looks something like this: [1.0, 0.0, 0.0]
    # order of the amounts matches with keys_filler  
    f1_wood_fiber_mass / filler_mass, 
    f2_hemp_shiv_mass / filler_mass, 
    f3_rice_husk_mass / filler_mass 
]

# main process 
main_process_exchanges_amounts = [ 
    # proportion of recipe ingredients (binder, filler) in the recipe 
    # looks something like this: [0.4, 0.6]
    # order of amounts matches keys_main_process 
    binder_mass / total_mass, 
    filler_mass / total_mass 
]

In [83]:
# exchanges amounts of main and sub-processes
# they all add up to one. 
print(binder_exchanges_amounts)
print(filler_exchanges_amounts)
print(main_process_exchanges_amounts)

[0.4, 0.5, 0.1]
[0.6, 0.3, 0.1]
[0.5, 0.5]


## Package keys and exchanges for LCA

In [84]:
# package sub-process keys and exchanges into lists for mapping 
# the order of sub-processes needs to match 

key_list_sub_processes = [keys_binder, keys_filler]
exchanges_list_sub_processes = [binder_exchanges_amounts, filler_exchanges_amounts]

print(key_list_sub_processes)
print('')
print(exchanges_list_sub_processes)

[{'b1_potato_starch_mass': '8ce68e5f71f8f8c5dc50edb913e9cea1', 'b2_lignin_resin_mass': '9ef63345ae414b0aa5683f09f4d0b806', 'b3_soy_resin_mass': '16a6869a94ad2aaf84d113c88657406b'}, {'f1_wood_fiber_mass': 'd82e94ec45e662362de31d0ac344dc51', 'f2_hemp_shiv_mass': 'b8b0f0ca4e8e1a99ddae3f6ae610be2e', 'f3_rice_husk_mass': '94c1cb9e760faf73987ffea6c08cfcf1'}]

[[0.4, 0.5, 0.1], [0.6, 0.3, 0.1]]


In [85]:
mapping_exchanges = create_mapping(keys_recipe, exchanges_list_sub_processes)
mapping_names = create_mapping(keys_recipe, key_list_sub_processes)
unique_activities = create_list_with_unique_activities(key_list_sub_processes)

print('mapping_exchanges: keys_recipe --> exchanges_list_sub_processes')
print(f'keys_recipe: \n{keys_recipe}')
print(f'exchanges_list_sub_processes: \n{exchanges_list_sub_processes}')
print('')
print('mapping_names: keys_recipe --> key_list_sub_processes')
print(f'keys_recipe: \n{keys_recipe}')
print(f'key_list_sub_processes: \n{key_list_sub_processes}')

mapping_exchanges: keys_recipe --> exchanges_list_sub_processes
keys_recipe: 
{'binder': '', 'filler': ''}
exchanges_list_sub_processes: 
[[0.4, 0.5, 0.1], [0.6, 0.3, 0.1]]

mapping_names: keys_recipe --> key_list_sub_processes
keys_recipe: 
{'binder': '', 'filler': ''}
key_list_sub_processes: 
[{'b1_potato_starch_mass': '8ce68e5f71f8f8c5dc50edb913e9cea1', 'b2_lignin_resin_mass': '9ef63345ae414b0aa5683f09f4d0b806', 'b3_soy_resin_mass': '16a6869a94ad2aaf84d113c88657406b'}, {'f1_wood_fiber_mass': 'd82e94ec45e662362de31d0ac344dc51', 'f2_hemp_shiv_mass': 'b8b0f0ca4e8e1a99ddae3f6ae610be2e', 'f3_rice_husk_mass': '94c1cb9e760faf73987ffea6c08cfcf1'}]


In [86]:
main_process_names = ['recipe production']
mapping_exchanges_mp = {main_process_names[0]: main_process_exchanges_amounts}
names_dictionary_sp = create_name_dictionaries(mapping_names, key_list_sub_processes)
names_dictionary_mp = {main_process_names[0]: list(names_dictionary_sp.keys())}

print('mapping_exchanges_mp: main_process_names --> main_process_exchanges_amounts')
print(f'main_process_names: \n{main_process_names}')
print(f'main_process_exchanges_amounts: \n{main_process_exchanges_amounts}\n')

print('names_dictionary: mapping_names --> key_list_sub_processes')
print(f'mapping_names: \n{mapping_names}')
print(f'key_list_sub_processes: \n{key_list_sub_processes}')
print(f'names_dictionary: \n{names_dictionary_sp}\n')

print(names_dictionary_mp)

mapping_exchanges_mp: main_process_names --> main_process_exchanges_amounts
main_process_names: 
['recipe production']
main_process_exchanges_amounts: 
[0.5, 0.5]

names_dictionary: mapping_names --> key_list_sub_processes
mapping_names: 
{'binder': ['8ce68e5f71f8f8c5dc50edb913e9cea1', '9ef63345ae414b0aa5683f09f4d0b806', '16a6869a94ad2aaf84d113c88657406b'], 'filler': ['d82e94ec45e662362de31d0ac344dc51', 'b8b0f0ca4e8e1a99ddae3f6ae610be2e', '94c1cb9e760faf73987ffea6c08cfcf1']}
key_list_sub_processes: 
[{'b1_potato_starch_mass': '8ce68e5f71f8f8c5dc50edb913e9cea1', 'b2_lignin_resin_mass': '9ef63345ae414b0aa5683f09f4d0b806', 'b3_soy_resin_mass': '16a6869a94ad2aaf84d113c88657406b'}, {'f1_wood_fiber_mass': 'd82e94ec45e662362de31d0ac344dc51', 'f2_hemp_shiv_mass': 'b8b0f0ca4e8e1a99ddae3f6ae610be2e', 'f3_rice_husk_mass': '94c1cb9e760faf73987ffea6c08cfcf1'}]
names_dictionary: 
{'binder': ['b1_potato_starch_mass', 'b2_lignin_resin_mass', 'b3_soy_resin_mass'], 'filler': ['f1_wood_fiber_mass', 'f2_h

## Calculate unit impacts

Here, `mpLCAer` **only** calculates the **unit impacts** of the individual bw processes we've linked, like 'market for potato starch'. We haven't added any amounts here yet, so `my_lca.unit_impacts` is just all the unit impacts of the sub-sub-processes added together. 

Later on, we will add the amounts for the sub-sub-processes, which will allow us to actually do an LCA. 

In [87]:
mapping_names

{'binder': ['8ce68e5f71f8f8c5dc50edb913e9cea1',
  '9ef63345ae414b0aa5683f09f4d0b806',
  '16a6869a94ad2aaf84d113c88657406b'],
 'filler': ['d82e94ec45e662362de31d0ac344dc51',
  'b8b0f0ca4e8e1a99ddae3f6ae610be2e',
  '94c1cb9e760faf73987ffea6c08cfcf1']}

In [88]:
my_lca = mpLCAer(4, methods_gp, brightway_configuration_dictionary)
my_lca.import_isolated_environment()
my_lca.lca_calculations(mapping_names)
my_lca.unit_impacts

{'binder': array([[2.07373188e-02, 1.65305088e+00, 6.09557667e+01, 2.23068055e-02,
         1.57998871e-07, 3.61765650e-06, 7.81788972e-08, 1.66614917e-03,
         1.73522884e-01],
        [7.38103379e-03, 3.18001977e+00, 3.04873567e+01, 8.42954009e-03,
         1.71471698e-07, 4.72702416e-07, 1.71751941e-04, 8.14884096e-04,
         1.09093991e-01],
        [1.17505421e-02, 3.32204571e+00, 4.59108830e+01, 1.50596106e-02,
         2.33231716e-07, 6.10519333e-07, 5.80978580e-08, 1.89500991e-03,
         1.85285092e-01]]),
 'filler': array([[ 3.81952738e-04,  8.68403279e-02,  1.06435213e+00,
          3.79057085e-04,  2.08621356e-08,  2.14857998e-08,
          1.85138641e-09,  5.74986885e-05,  9.40449764e-03],
        [ 1.05259764e-03,  1.35887654e-01,  8.78202245e+00,
          9.10640730e-03,  7.51021308e-09, -9.29508414e-07,
          2.60270738e-09,  3.31062043e-04,  1.49521109e-02],
        [ 3.19482111e-06,  7.35097340e-04,  7.26436821e-03,
          7.46989860e-07,  5.42529968e-1

## Calculate LCA with exchange amounts

In [89]:
# HANDLE SUB-PROCESSES (sp = sub-process)

# calculate impact of each sub-process (binder, filler)
# based on exchanges with sub-sub-processes (b1_potato_starch, etc) and unit impacts 
sp_exchange_manager = ExchangeHolder(methods_gp) 
sp_exchange_manager.create_exchange_arrays(mapping_exchanges) # manage exchanges from sub-sub-processes to sub-processes
sp_impact_calculator = ImpactCalculator()
sp_impact_calculator.impact_calculation_simple(my_lca.unit_impacts, sp_exchange_manager.exchanges_dict) # calculate unit impacts based on exchanges

# store impact results for sub-processes 
sp_results_handler = ImpactHandler(impact_categories_names)
sp_results_handler.create_dataframes(sp_impact_calculator.simple_impacts, names_dictionary_sp)


In [90]:
mp_results_handler.df_impacts['recipe production']

,AP,GWP100,ECFW,EP,HTC,HTNC,ODP,PMFP,MIR
binder,0.006580,1.291717,22.108537,0.007322,8.612928e-08,8.722329e-07,4.295653e-05,0.000632,0.071242
filler,0.000273,0.046472,1.636972,0.001480,7.387885e-09,-1.329718e-07,9.465283e-10,0.000067,0.005069
Sum,0.006853,1.338189,23.745509,0.008801,9.351717e-08,7.392611e-07,4.295747e-05,0.000699,0.076311


In [91]:
# HANDLE MAIN PROCESS (mp = main process)

# scale sub-process unit results for sub-processes with exchange amounts for main process
mp_results_handler = ImpactHandler(impact_categories_names)
mp_results_handler.calculate_total_unit_impact(sp_results_handler.total_impact_arrays, main_process_names[0])

# calculate unit impact of main process (recipe)
# based on exchanges with sub-processes (binder, filler)
mp_exchange_manager = ExchangeHolder(methods_gp) # biosphere exchanged based on pre-defined impact calculation method (e.g. TRACI)
mp_exchange_manager.create_exchange_arrays(mapping_exchanges_mp) # manage exchanges from sub-processes to main process
mp_impact_calculator = ImpactCalculator()
mp_impact_calculator.impact_calculation_simple(mp_results_handler.total_unit_impact, mp_exchange_manager.exchanges_dict)

# store impact results for main process 
mp_results_handler.create_dataframes(mp_impact_calculator.simple_impacts, names_dictionary_mp)


In [92]:
# SUMMARIZE AND SAVE RESULTS 

# create dictionary of dfs with contribution analysis results 
df_list_results = list(sp_results_handler.df_contributions.values())
df_list_results.append(mp_results_handler.df_contributions['recipe production'])
sheet_names = list(keys_recipe.keys()) + ['Total'] # Names of excel sheet for storage
dict_with_contribution_dfs = create_dataframe_dict(df_list_results, sheet_names)

# create list of dfs with sub-process and main-process results 
# and put in dictionary 
list_with_absolute_results = list(sp_results_handler.df_impacts.values())
list_with_absolute_results.append(mp_results_handler.df_impacts['recipe production'])
dict_with_absolute_results_df = create_dataframe_dict(list_with_absolute_results, sheet_names)

# save results in excel 
excel_file_path_contribution_results = os.path.join(target_dir, 'recipeProduction_contribution.xlsx')
excel_file_path_absolute_results = os.path.join(target_dir, 'recipeProduction_absoloute.xlsx')
save_dataframes_to_excel(dict_with_contribution_dfs, excel_file_path_contribution_results)
save_dataframes_to_excel(dict_with_absolute_results_df, excel_file_path_absolute_results)

Excel file 'results/lca/recipeProduction_contribution.xlsx' saved successfully.
Excel file 'results/lca/recipeProduction_absoloute.xlsx' saved successfully.


In [93]:
df_list_results[1]

,AP,GWP100,ECFW,EP,HTC,HTNC,ODP,PMFP,MIR
f1_wood_fiber_mass,42.028989,56.059773,19.505868,7.685067,84.714914,-4.847449,58.679275,25.771875,55.663600
f2_hemp_shiv_mass,57.912420,43.861137,80.471944,92.312409,15.248368,104.854009,41.246112,74.193777,44.249483
f3_rice_husk_mass,0.058591,0.079090,0.022188,0.002524,0.036718,-0.006560,0.074613,0.034348,0.086917


## Add extra main process - transportation

1 kg biopol wall panel = 1 kg biopol recipe + 10 km transportation

The process hierarchy looks something like this. I'm not sure if each main process **must** have three layers, but that's the standard for lcpy, so I'll just do that for now. 
- wall_panel
    - recipe
        - binder
            - b1
            - b2 
            - b3 
        - filler
            - f1
            - f2
            - f3 
    - transportation 
        - truck
            - truck1
            - truck2
            - truck3
        - train
            - train1
            - train2
            - train3

### Transportation unit impacts
- here, we create a new LCA object `my_lca_wall_panel` for a biopol wall panel, which includes two main processes: transportation and recipe production. 
- right now, we calculate the unit impacts for **only** transportation. Later, we will add the recipe production process to the new LCA object. 

In [94]:
# model other sub-process(s) - in our case, transportation 
key_transportation = {
    'transportation': '9f5ff656ecdc15acd23085211786936f' 
}
exchange_list_transportation = [10] # 10km of transportation = 1kg of printed wall panel

# model new main process - biopol wall panel (1 kg) 
key_wall_panel = {
    'transportation': '', 
}
exchange_list_wall_panel = [1]

In [95]:
key_list_wall_panel = [key_transportation]
exchanges_list_wall_panel = [exchange_list_transportation]

mapping_wall_panel_names = create_mapping(key_wall_panel, key_list_wall_panel)
unique_activities_wall_panel = create_list_with_unique_activities(key_list_wall_panel)
mapping_wall_panel_exchanges = create_mapping(key_wall_panel, exchanges_list_wall_panel)

my_lca_wall_panel = mpLCAer(4, methods_gp, brightway_configuration_dictionary)
my_lca_wall_panel.import_isolated_environment()
my_lca_wall_panel.lca_calculations(mapping_wall_panel_names)

In [96]:
my_lca_wall_panel.unit_impacts

{'transportation': array([[4.16404006e-04, 1.41896635e-01, 1.28912783e+00, 1.14120326e-04,
         1.03015172e-08, 3.31454746e-08, 3.34497549e-09, 7.58738798e-05,
         1.08835144e-02]])}

In [97]:
print(mapping_wall_panel_names) # mapping_wall_panel
print(mapping_wall_panel_exchanges) # mapping_each_wall_panel 
print(my_lca_wall_panel.unit_impacts)

{'transportation': ['9f5ff656ecdc15acd23085211786936f']}
{'transportation': [10]}
{'transportation': array([[4.16404006e-04, 1.41896635e-01, 1.28912783e+00, 1.14120326e-04,
        1.03015172e-08, 3.31454746e-08, 3.34497549e-09, 7.58738798e-05,
        1.08835144e-02]])}


### Append recipe production process
- unit impacts 
- mapping_wall_panel_names
- mapping_wall_panel_exchanges
- key_list_wall_panel
- exchanges_list_wall_panel

In [98]:
my_lca_wall_panel.unit_impacts['recipe production'] = mp_results_handler.total_impact_arrays['recipe production'].reshape(1, len(impact_categories_names))
mapping_wall_panel_names['recipe production'] = ['']
mapping_wall_panel_exchanges['recipe production'] = [1.] # in case of multiple scenarios and time_steps the 1. should be an array
key_list_wall_panel.append({'recipe production': ''})
exchanges_list_wall_panel.append([1.])

key_wall_panel['recipe production'] = ['']
exchange_list_wall_panel.append(1.)

In [99]:
my_lca_wall_panel.unit_impacts

{'transportation': array([[4.16404006e-04, 1.41896635e-01, 1.28912783e+00, 1.14120326e-04,
         1.03015172e-08, 3.31454746e-08, 3.34497549e-09, 7.58738798e-05,
         1.08835144e-02]]),
 'recipe production': array([[6.85288452e-03, 1.33818941e+00, 2.37455089e+01, 8.80144222e-03,
         9.35171696e-08, 7.39261073e-07, 4.29574724e-05, 6.98633256e-04,
         7.63109006e-02]])}

### LCA modeling with two main processes

In [100]:
sp_exchange_wall_panel = ExchangeHolder(methods_gp)
sp_impact_wall_panel = ImpactCalculator()

sp_exchange_wall_panel.create_exchange_arrays(mapping_wall_panel_exchanges)
sp_impact_wall_panel.impact_calculation_simple(my_lca_wall_panel.unit_impacts, sp_exchange_wall_panel.exchanges_dict)
sp_handler_wall_panel = ImpactHandler(impact_categories_names)
names_dictionary_wall_panel = create_name_dictionaries(mapping_wall_panel_names, key_list_wall_panel)
sp_handler_wall_panel.create_dataframes(sp_impact_wall_panel.simple_impacts, names_dictionary_wall_panel)
total_handler_wall_panel = ImpactHandler(impact_categories_names)

list_with_names_of_wall_panel = ['wall panel production']
total_handler_wall_panel.calculate_total_unit_impact(sp_handler_wall_panel.total_impact_arrays, list_with_names_of_wall_panel[0])
total_exchanger_wall_panel = ExchangeHolder(methods_gp)
total_impact_calculator_wall_panel = ImpactCalculator()

mapping_exch_total_wall_panel = {list_with_names_of_wall_panel[0]: exchange_list_wall_panel}
names_dictionary_wall_panel_main = {list_with_names_of_wall_panel[0]: list(names_dictionary_wall_panel.keys()) }

total_exchanger_wall_panel.create_exchange_arrays(mapping_exch_total_wall_panel)
total_impact_calculator_wall_panel.impact_calculation_simple(total_handler_wall_panel.total_unit_impact, total_exchanger_wall_panel.exchanges_dict)
total_handler_wall_panel.create_dataframes(total_impact_calculator_wall_panel.simple_impacts, names_dictionary_wall_panel_main)

df_list_wall_panel = list(sp_handler_wall_panel.df_contributions.values())
df_list_wall_panel.append(total_handler_wall_panel.df_contributions['wall panel production'])

sheet_names_wall_panel = list(key_wall_panel.keys()) + ['Total']
dict_with_dfs_contr_wall_panel = create_dataframe_dict(df_list_wall_panel, sheet_names_wall_panel)
df_list_total_wall_panel = list(sp_handler_wall_panel.df_impacts.values())
df_list_total_wall_panel.append(total_handler_wall_panel.df_impacts['wall panel production'])
dict_with_dfs_total_wall_panel = create_dataframe_dict(df_list_total_wall_panel, sheet_names_wall_panel)

excel_file_path_contr_wall_panel = os.path.join(target_dir, 'wall_panel_contributions.xlsx')
save_dataframes_to_excel(dict_with_dfs_contr_wall_panel, excel_file_path_contr_wall_panel)

excel_file_path_total_wall_panel = os.path.join(target_dir, 'wall_panel_totals.xlsx')
save_dataframes_to_excel(dict_with_dfs_total_wall_panel, excel_file_path_total_wall_panel)
sp_handler_wall_panel.contribution_to_total_impact_per_sub_sub_processes(dict_with_dfs_total_wall_panel, dict_with_dfs_contr_wall_panel, unique_activities_wall_panel, name= 'wall panel production', target_dir = target_dir)


Excel file 'results/lca/wall_panel_contributions.xlsx' saved successfully.
Excel file 'results/lca/wall_panel_totals.xlsx' saved successfully.


In [20]:
mapping_exch_total_wall_panel

{'wall panel production': [1, 1.0]}

# Integrate into optimization 

In [101]:
my_lca_wall_panel.unit_impacts

{'transportation': array([[4.16404006e-04, 1.41896635e-01, 1.28912783e+00, 1.14120326e-04,
         1.03015172e-08, 3.31454746e-08, 3.34497549e-09, 7.58738798e-05,
         1.08835144e-02]]),
 'recipe production': array([[6.85288452e-03, 1.33818941e+00, 2.37455089e+01, 8.80144222e-03,
         9.35171696e-08, 7.39261073e-07, 4.29574724e-05, 6.98633256e-04,
         7.63109006e-02]])}

In [102]:
# --- imports ---
import numpy as np
import pandas as pd
import rasterio
from rasterio.mask import mask
from shapely.geometry import Point

# --- user inputs (edit these) ---
raster_path = "data/processed/EU_crop_map_yield.tif" # band 1 = crop code, band 3 = yield
conversion_csv = "data/processed/bipol_ranges_with_feedstocks.csv" # must have columns: id, option, eu_crop_code
x = 3537560.0 # factory X in raster CRS (meters if projected)
y = 2791250.0 # factory Y in raster CRS
buffer_radius = 5000 # meters
nodata = None # set to a value if your raster uses one (e.g., 0 or -9999)

# --- load conversion table ---
conversion = pd.read_csv(conversion_csv)

# --- geometry for analysis area ---
factory_point = Point(x, y)
buffer_geom = factory_point.buffer(buffer_radius)  # assumes projected CRS (meters)

# --- crop raster to buffer and compute total yield per crop code ---
with rasterio.open(raster_path) as src:
    # ensure geometry is GeoJSON-like
    out_image, _ = mask(src, [buffer_geom.__geo_interface__], crop=True, indexes=[1, 3])

crop_band = out_image[0]
yield_band = out_image[1]

if nodata is not None:
    valid = (crop_band != nodata) & (yield_band != nodata)
    crop_band = crop_band[valid]
    yield_band = yield_band[valid]

# dict: {crop_code -> total_yield_in_buffer}
unique_crops = np.unique(crop_band)
total_yield_per_crop = {
    int(c): float(yield_band[crop_band == c].sum())
    for c in unique_crops
}

# --- map crop yields to ingredient availability (2D arrays) ---
ingredient_availability = {}
for _, row in conversion.iterrows():
    key = f"{row['id']}_{row['option']}_mass"
    crop_code = row['eu_crop_code']
    value = total_yield_per_crop.get(int(crop_code), 0.0)
    ingredient_availability[key] = np.array([[value]])  # keep 2D shape

# ensure required keys exist (fill with zeros if missing)
required_keys = [
    "b1_potato_starch_mass",
    "b2_lignin_resin_mass",
    "b3_soy_resin_mass",
    "f1_wood_fiber_mass",
    "f2_hemp_shiv_mass",
    "f3_rice_husk_mass",
]
for k in required_keys:
    if k not in ingredient_availability:
        ingredient_availability[k] = np.array([[0.0]])

# --- transportation distance (placeholder) ---
transportation_distance = 5000  # TODO: replace with real distance calc if needed

# --- impact calculation (uses your external LCA objects/functions) ---
# extract 2D arrays
b1 = ingredient_availability["b1_potato_starch_mass"]
b2 = ingredient_availability["b2_lignin_resin_mass"]
b3 = ingredient_availability["b3_soy_resin_mass"]
f1 = ingredient_availability["f1_wood_fiber_mass"]
f2 = ingredient_availability["f2_hemp_shiv_mass"]
f3 = ingredient_availability["f3_rice_husk_mass"]

# combine elementwise
binder_mass = b1 + b2 + b3
filler_mass = f1 + f2 + f3
total_mass = binder_mass + filler_mass

# guard against divide-by-zero
eps = 1e-12
binder_mass_safe = binder_mass + (binder_mass == 0) * eps
filler_mass_safe = filler_mass + (filler_mass == 0) * eps
total_mass_safe  = total_mass  + (total_mass  == 0) * eps

binder_fracs = [b1 / binder_mass_safe, b2 / binder_mass_safe, b3 / binder_mass_safe]
filler_fracs = [f1 / filler_mass_safe, f2 / filler_mass_safe, f3 / filler_mass_safe]
mix_fracs = [binder_mass_safe / total_mass_safe, filler_mass_safe / total_mass_safe]

# exchanges/mapping per your framework (placeholders come from your environment)
exchanges_list_sp = [binder_fracs, filler_fracs]
mapping_exchanges = create_mapping(keys_recipe, exchanges_list_sp)

calc = fast_calculator()
calc.calculation_static_lcia(mapping_exchanges, my_lca.unit_impacts)

# NOTE: transportation_distance is not yet integrated into LCA here; add when you wire it up
calc.calculation_impact_senarios(mix_fracs, calc.impact, "wall panel production")

# --- final result ---
total_impact = calc.total_impact["wall panel production"][:, :, 0].T

# if you want to peek:
print("total_yield_per_crop:", total_yield_per_crop)
print("ingredient_availability keys:", list(ingredient_availability.keys()))
print("total_impact shape:", total_impact.shape)
total_impact


total_yield_per_crop: {-9999: -539626048.0, 100: 2140190.0, 211: 4076640.0, 213: -4299570.0, 216: 2558080.0, 221: -239976.0, 222: -679932.0, 231: 221100.0, 232: -8529147.0, 250: -2379762.0, 300: 15201750.0, 500: 16288580.0, 700: -34546544.0}
ingredient_availability keys: ['b1_potato_starch_mass', 'b2_lignin_resin_mass', 'b3_soy_resin_mass', 'f1_wood_fiber_mass', 'f2_hemp_shiv_mass', 'f3_rice_husk_mass']
total_impact shape: (1, 9)


array([[9.37097672e-03, 1.16002461e+00, 2.93674811e+01, 1.07484770e-02,
        9.77881968e-08, 1.38813806e-06, 1.73273367e-05, 8.51882643e-04,
        9.06623769e-02]])

In [103]:
import numpy as np

def to_mapping_format(d):
    out = {}
    for k, vals in d.items():
        # make sure we can iterate
        if not isinstance(vals, (list, tuple, np.ndarray)):
            vals = [vals]

        converted = []
        for v in vals:
            if isinstance(v, np.ndarray):
                a = v
                # force 2D [[...]]
                if a.ndim == 0:
                    a = np.array([[a.item()]])
                elif a.ndim == 1:
                    a = a.reshape(1, a.size)
                # if it's already 2D, keep it
            else:
                a = np.array([[float(v)]])
            converted.append(a)
        out[k] = converted
    return out

# example use
mapping_wall_panel_exchanges = {'transportation': [1], 'recipe production': [1.0]}

mapping_wall_panel_exchanges_2d = to_mapping_format(mapping_wall_panel_exchanges)

print(mapping_wall_panel_exchanges_2d)
# {'transportation': [array([[1.]])], 'recipe production': [array([[1.]])]}


{'transportation': [array([[1.]])], 'recipe production': [array([[1.]])]}


In [105]:
mix_fracs = [np.array([[10.]]), np.array([[1.]])]
mapping_exchanges = to_mapping_format(mapping_wall_panel_exchanges)

calc = fast_calculator()
calc.calculation_static_lcia(mapping_exchanges, my_lca_wall_panel.unit_impacts)
calc.calculation_impact_senarios(mix_fracs, calc.impact, "wall panel production")
total_impact = calc.total_impact["wall panel production"][:, :, 0].T

total_impact

array([[1.10169246e-02, 2.75715576e+00, 3.66367872e+01, 9.94264548e-03,
        1.96532342e-07, 1.07071582e-06, 4.29909222e-05, 1.45737205e-03,
        1.85146045e-01]])

## Check results
Does the total_impact for wall production make sense? 
- unit impacts: are the unit impacts for wall panel production the sum of unit impacts for transportation and recipe production? 
- total impact: is the total impact for wall panel production the sum of total impacts for transportation and recipe production? 
    - if we change the exchange amounts, do the total impacts change accordingly? 

In [37]:
# check unit impacts 
print(my_lca_wall_panel.unit_impacts) # unit impacts for wall panel (transportation + recipe production)
print(my_lca.unit_impacts) # unit impacts for recipe production (binder + filler)

{'transportation': array([[4.16404006e-04, 1.41896635e-01, 1.28912783e+00, 1.14120326e-04,
        1.03015172e-08, 3.31454746e-08, 3.34497549e-09, 7.58738798e-05,
        1.08835144e-02]]), 'recipe production': array([[6.85288452e-03, 1.33818941e+00, 2.37455089e+01, 8.80144222e-03,
        9.35171696e-08, 7.39261073e-07, 4.29574724e-05, 6.98633256e-04,
        7.63109006e-02]])}
{'binder': array([[2.07373188e-02, 1.65305088e+00, 6.09557667e+01, 2.23068055e-02,
        1.57998871e-07, 3.61765650e-06, 7.81788972e-08, 1.66614917e-03,
        1.73522884e-01],
       [7.38103379e-03, 3.18001977e+00, 3.04873567e+01, 8.42954009e-03,
        1.71471698e-07, 4.72702416e-07, 1.71751941e-04, 8.14884096e-04,
        1.09093991e-01],
       [1.17505421e-02, 3.32204571e+00, 4.59108830e+01, 1.50596106e-02,
        2.33231716e-07, 6.10519333e-07, 5.80978580e-08, 1.89500991e-03,
        1.85285092e-01]]), 'filler': array([[ 3.81952738e-04,  8.68403279e-02,  1.06435213e+00,
         3.79057085e-04,  2.0

In [30]:
exchanges_list_sub_processes

[[0.4, 0.5, 0.1], [0.6, 0.3, 0.1]]

In [46]:
main_process_exchanges_amounts

[0.5, 0.5]

In [44]:
import numpy as np

# Example
unit_impacts = my_lca.unit_impacts
exchanges = exchanges_list_sub_processes

# Get keys in consistent order
keys = list(unit_impacts.keys())

# Multiply each array by the corresponding exchange list
scaled_impacts = {
    key: unit_impacts[key] * np.array(exchanges[i])[:, np.newaxis]
    for i, key in enumerate(keys)
}
total_impacts = {
    key: np.sum(scaled_impacts[key], axis=0)
    for key in keys
}

print(sum(total_impacts.values()))
print(my_lca_wall_panel.unit_impacts['recipe production'])

[1.37057690e-02 2.67637881e+00 4.74910178e+01 1.76028844e-02
 1.87034339e-07 1.47852215e-06 8.59149448e-05 1.39726651e-03
 1.52621801e-01]
[[6.85288452e-03 1.33818941e+00 2.37455089e+01 8.80144222e-03
  9.35171696e-08 7.39261073e-07 4.29574724e-05 6.98633256e-04
  7.63109006e-02]]
